# 04 · GitHub 与自动检查：正本永远是好的

## 这一章在整个项目里的位置

```
00 总览 → 01 存档旧项目 ✅ → 02 新项目骨架 ✅ → 03 自动检查 ✅ → [04 GitHub 与 CI ✅] → 05 准备考题 ✅ → 06 评测集 ✅ → 07 判分 ✅ → 08 MLflow 评测 ✅ → 09 第一个基线 ✅ → 10 第一个工具 ✅ → 11 第二个工具 ✅ → 12 起：更多工具、agent …
```

> **这一章 10 月 7 日已经做完了。** 下面的命令只是从 GitHub 上查看现状。

## 背景：代码只在自己电脑上，有三个问题

1. **硬盘坏了就没了，别人也看不到。** 旧项目 8 月那 5 次提交从来没推到 GitHub，01 号要专门去救，就是这个原因
2. **03 号的检查只装在你这台电脑上。** 换一台电脑，或者忘了装，就没有了；还可以用 `--no-verify` 跳过
3. **"在我电脑上能跑"不等于别人能跑。** 你电脑上有的文件、装过的包，别人那里不一定有

**公司里的做法**：正本放在 GitHub；每个改动先在分支上做，提一个"合并请求"；GitHub 在一台**全新的电脑**上把检查全部重跑一遍；绿了、有人看过，才能并入正本：

```
在分支上改 → 推到 GitHub → 提合并请求 → GitHub 自动检查（CI）→ 你看过，点合并 → 正本
```

## 这一章做的 4 件事

| # | 做什么 | 用的技术 | 解决什么问题 |
|---|---|---|---|
| 1 | 正本放到 GitHub | GitHub 私有仓库 | 不怕丢，能协作 |
| 2 | 每个改动自动检查 | GitHub Actions（CI） | 不管谁、从哪台电脑交的都要过 |
| 3 | 改动走合并请求 | Pull Request + squash 合并 | 先检查、再并入；正本历史一行一个改动 |
| 4 | 看自动检查拦住的问题 | CI 的运行记录 | 问题在进入正本之前被发现 |

**花费：0。** GitHub 私有仓库和自动检查对个人免费（有每月的免费额度）。我们每次检查约 20 秒，用不完。

---

## 准备

运行下面这一格。它定义一个叫 `run` 的小工具，用来执行命令并显示结果。**不用看代码。**

In [ ]:
import os, subprocess
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
os.environ.update(NO_COLOR="1", GH_PAGER="cat", GIT_PAGER="cat")
def run(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout + r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f"failed: {cmd}")
print("项目目录:", ROOT)

---

## 第 1 步 · 正本放到 GitHub

### 在干什么
在 GitHub 上建一个**私有**仓库 `pipi0616/rare-disease-agent`，把本地的正本推上去。从此 GitHub 上的是正本，你电脑上的是工作副本。

### 背景
见上面的问题 1。

### 用了什么技术，为什么

| 词 | 是什么 | 比喻 |
|---|---|---|
| **GitHub** | 存放 git 仓库的网站，团队在上面协作、审查代码 | 实验室的档案柜 |
| **远程仓库**（remote，名字叫 `origin`） | GitHub 上的那一份 | 档案柜里的正本 |
| **推送 / 拉取**（push / pull） | 把本地的提交传上去 / 把 GitHub 上的新提交拿下来 | 交档案 / 取档案 |

当时运行的命令（已经做过）：

```bash
gh repo create rare-disease-agent --private --source=. --remote=origin --push
```

`gh` 是 GitHub 官方的命令行工具，网页上能做的事，它都能做。

**为什么先建私有**：项目还没成形。到路线第 5 步做成服务、能展示的时候，再改成公开放进简历，之前的记录会原样带过去。

### 公司里什么情况怎么做
- **当天的工作当天推**：本地只是工作副本，正本在服务器上
- **仓库公开之前**：检查提交记录里的作者邮箱，能看到仓库的人都能看到它。个人公开仓库的常见做法是用 GitHub 提供的隐私邮箱。我们最早的 3 次提交用的是你的 Gmail，公开前再处理

### 执行

In [ ]:
run("git remote -v")
run("""gh repo view --json nameWithOwner,visibility,url --jq '.nameWithOwner + "  " + .visibility + "  " + .url'""")

### 结果怎么看
- `origin  https://github.com/pipi0616/rare-disease-agent.git`：本地知道正本在哪
- `pipi0616/rare-disease-agent  PRIVATE  https://...`：私有仓库，只有你能看。可以点开那个网址看看

### 面试怎么说
> "The source of truth is the GitHub repository; local clones are working copies, and work is pushed daily."

---

## 第 2 步 · 每个改动自动检查：CI

### 在干什么
每次有改动推到 GitHub，GitHub 自动拿一台**全新的电脑**，从零把项目装一遍，再跑一遍 03 号的全部检查。

### 背景
CI 就是**自动复现**。比喻：别的实验室按你的实验记录重做一遍，看能不能得到同样的结果；CI 是你每改一次代码，就自动重做一次。

它能发现你在自己电脑上发现不了的问题。典型例子：你写了新代码，自己电脑上测试都通过了，但**忘了把一个文件提交上去**。03 号的检查看不出来（文件就在你电脑上），只有在全新的电脑上会失败。这件事在我们项目里真的发生过，见第 4 步。

### 用了什么技术，为什么

| 词 | 是什么 |
|---|---|
| **CI**（持续集成，Continuous Integration） | 每次推送代码，自动运行检查和测试 |
| **GitHub Actions** | GitHub 自带的 CI 系统 |
| **workflow** | 一份 CI 配置：什么时候跑、跑什么 |

配置在 [`.github/workflows/ci.yml`](../../.github/workflows/ci.yml)，只有 40 行：

| 部分 | 做什么 |
|---|---|
| 什么时候跑 | 每个合并请求、每次推到 `main` |
| 在哪跑 | GitHub 的一台全新 Ubuntu 电脑 |
| 装环境 | `uv sync --locked`：严格按锁文件装，锁文件过期直接报错 |
| 检查 | ruff → mypy → pytest，和 03 号一样 |

每次约 20 秒。失败了，GitHub 会给你发邮件。

**pre-commit 和 CI 是不是重复了？** 比喻：pre-commit 是**出门前自己检查行李**，CI 是**机场安检**。

| | pre-commit（03 号） | CI（这一章） |
|---|---|---|
| 什么时候跑 | `git commit` 的那一刻 | 推到 GitHub 之后 |
| 在哪跑 | 你的电脑 | GitHub 的全新电脑 |
| 能不能跳过 | 能（`--no-verify`，或者根本没装） | 不能，结果所有人都看得到 |
| 擅长什么 | 快；拦住**绝不能离开你电脑的东西**：密钥、大文件 | 发现"在我电脑上能跑"的问题 |

两边都跑 ruff、mypy、pytest：pre-commit 让你早发现，CI 保证谁也绕不过去。密钥只能靠 pre-commit 拦：等 CI 发现，密钥已经在 GitHub 上了。

### 公司里什么情况怎么做
- **CI 变红**：点进去，找日志里带 `##[error]` 的那一行，通常它就是答案。在本地修好，再推一次。**红的不合并**，这是规矩
- **本地通过、CI 失败**：多半是忘了提交某个文件，或者加了包没更新锁文件。这正是 CI 存在的意义

### 执行
看最近 5 次自动检查：

In [ ]:
run("""gh run list --limit 5 --json conclusion,headBranch,createdAt --jq '.[] | .createdAt[:16] + "  " + .conclusion + "  " + .headBranch'""")

### 结果怎么看
每一行：时间、结果、哪个分支。`success` 就是网页上的绿勾。

> 想在网页上看：打开仓库 → 顶部 **Actions** 标签，点进任意一次，能看到每一步的日志。

### 面试怎么说
> "Every push and pull request runs lint, type checks and tests on a clean runner with the locked dependencies; nothing merges unless it's green."

---

## 第 3 步 · 改动走合并请求

### 在干什么
改动不直接进正本：先在分支上做，推到 GitHub，提一个**合并请求**；CI 的结果挂在上面；你看过、点"合并"，才进入正本。

### 背景
只有自动检查还不够：检查只能发现"坏了"，发现不了"做得不对"（比如方向改错了）。公司里每个改动都要有第二个人看过。

### 用了什么技术，为什么
**合并请求**（Pull Request，简称 PR）："我在分支上改了这些，请检查后并入正本"。

我们每一步的固定流程：

```
我在分支上写代码 → 推到 GitHub → 开 PR → CI 自动检查 → 你看 Files changed → 你点 Squash and merge
```

仓库的合并方式设成了**只允许 squash 合并，合并后自动删分支**：
- **squash**：分支上的好几次提交压成一条，并入正本。正本的历史里，**一行就是一个完整的改动**，干净好读
- **自动删分支**：合并完的分支没用了，自动清掉

> **免费账号的限制**：公司里会给正本加一条硬规定："必须经过 PR、CI 必须是绿的，才能合并"（叫分支保护）。GitHub 免费账号的私有仓库不开放这个功能，所以现在靠流程自觉；仓库公开后就能打开。

### 公司里什么情况怎么做
- **一个 PR 只做一件事**，改动控制在几百行以内，审查的人才看得过来
- **PR 的说明写清楚**：改了什么、为什么、怎么验证
- **审查时看三样**：CI 是不是绿的、改了哪些文件、说明和实际改动是否一致

### 执行
列出所有合并请求：

In [ ]:
run("""gh pr list --state all --json number,state,title --jq '.[] | "#" + (.number|tostring) + "  " + .state + "  " + .title'""")

### 结果怎么看
每一行是一个改动。`MERGED` = 已经并入正本；`OPEN` = 还在等你点合并。

### 面试怎么说
> "All changes go through pull requests: CI must pass and the diff is reviewed before a squash merge, so main's history is one commit per change."

---

## 第 4 步 · 自动检查拦住的两次问题

### 在干什么
看 CI 的失败记录。这两次都是真实发生的，都在进入正本之前被拦住了。

### 执行

In [ ]:
run("""gh run list --status failure --json createdAt,headBranch --jq 'reverse | .[] | .createdAt[:16] + "  " + .headBranch'""")

### 结果怎么看
两行，按时间从早到晚：

**第一次**（`ci/github-actions` 分支）：CI 配置刚写好，第一次运行 3 秒就失败了。日志里只有一行是重点：

```
##[error]Unable to resolve action `astral-sh/setup-uv@v10`, unable to find version `v10`
```

意思是：配置里写的一个工具版本号 `v10` 不存在（它只有 `v10.2.0` 这种精确的版本号）。改成 `v10.2.0`，15 秒通过。

**第二次**（`data/phenopacket-store` 分支）：05 号的数据代码第一次提合并请求，测试失败：

```
ModuleNotFoundError: No module named 'raredx.data'
```

意思是：代码文件夹 `src/raredx/data/` 根本没被提交。原因就是 02 号说的 `.gitignore` 那个坑，`data/` 把它挡住了。**在你的电脑上，这个文件夹一直都在**，所以本地的检查全部通过；GitHub 在全新的电脑上一跑，立刻发现缺东西。

第二次正好说明了 CI 存在的意义：**"在我电脑上能跑"不算数。**

### 公司里什么情况怎么做
- **收到 CI 失败的邮件**：点进去找 `##[error]` 那一行，修好再推。红的那次不会进入正本
- **事后**：把踩过的坑写下来（我们写在这本笔记里），下次不再犯

### 面试怎么说
> "CI caught a file that was missing from a commit because of an over-broad .gitignore rule — everything passed locally because the file existed on my machine. That's exactly the class of bug a clean runner exists to catch."

---

## 本章小结

| 步骤 | 技术 | 公司里的作用 |
|---|---|---|
| 1 正本上 GitHub | GitHub 私有仓库 | 不怕丢，能协作 |
| 2 自动检查 | GitHub Actions（CI） | 谁也绕不过去的统一检查 |
| 3 合并请求 | PR + squash 合并 | 先检查、再审查、再并入 |
| 4 拦住问题 | CI 运行记录 | "在我电脑上能跑"不算数 |

**一句话**：从这一章起，每个改动都要在一台干净的电脑上过检查、经过你点合并，才能进入正本。

## 面试题

1. **Walk me through your development workflow.**
   > "Work happens on a branch; pre-commit runs lint, type checks and secret detection on every commit; I push and open a pull request; GitHub Actions runs the full checks on a clean runner with locked dependencies; after review it's squash-merged into main."

2. **Why do you need CI if everyone runs the checks locally?**
   > "Local hooks can be skipped or not installed, and a laptop has files and packages a fresh machine doesn't. CI is the one gate nobody can bypass, and it proves the project builds from the repository alone."

3. **CI is red on your pull request. What do you do?**
   > "Open the failed job, find the error line, reproduce locally, fix and push again. Red never merges."

**下一章 05 · 准备考题**：下载一万例真实病例，整理成统一格式，分成平时用的和最后考的两半。